# Umbrella Sampling Example

In this notebook we will progressively populate the windows of an Umbrella Sampling simulation starting from a starting configuration in a metastable state.
For our example we will use capped alanine dipeptide and sample along the ψ dihedral, but it should be easy enough to adopt this example to other molecular systems and sampling coordinates.

In [ ]:
import asyncio
import os
import numpy as np
import asyncmd
import asyncmd.gromacs
import asyncmd.trajectory

In [ ]:
class UmbrellaSamplingWindow:
    def __init__(self, window_center, seed_trajectories_generator, workdir,
                 progress_cv,  # we expect this to be a wrapped CV returning an 1d-array of shape (len(traj),)
                 n_steps_initial, n_steps_total,
                 engine_cls, engine_kwargs, walltime_per_part,
                 ):
        assert n_steps_initial <= n_steps_total
        self.window_center = window_center
        self.seed_trajectories_generator = seed_trajectories_generator
        self.workdir = workdir
        self.progress_cv = progress_cv
        self.n_steps_initial = n_steps_initial
        self.n_steps_total = n_steps_total
        self._first_trajectories = None
        self._all_trajectories = None
        self._propagator = asyncmd.trajectory.InPartsTrajectoryPropagator(
                                            n_steps=self.n_steps_initial,
                                            engine_cls=engine_cls,
                                            engine_kwargs=engine_kwargs,
                                            walltime_per_part=walltime_per_part,
                                            )

    async def await_first_trajectories(self):
        while self._first_trajectories is None:
            await asyncio.sleep(10)
        return self._first_trajectories

    async def generate_first_trajectories(self):
        # return the first trajectories running for n_steps_initial steps
        if self._first_trajectories is not None:
            return self._first_trajectories
        initial_conf = await self._get_initial_conf()
        print(f"Window at {self.window_center}: Initial configuration written.")
        trajectories = self._propagator.propagate(
                                            starting_configuration=initial_conf,
                                            workdir=self.dirname,
                                            deffnm="us",
                                            )
        print(f"Window at {self.window_center}: First trajectories done.")
        self._first_trajectories = trajectories
        return trajectories

    async def _get_initial_conf(self):
        # wait for seed trajectories, then find conf closest to window center,
        # write this initial conf to workdir and return it
        seed_trajs = await self.seed_trajectories_generator()
        extractor = asyncmd.trajectory.convert.NoModificationFrameExtractor()
        # TODO/FIXME: use all trajectories!
        cv_vals = await self.progress_cv(seed_trajs[-1])
        idx = np.argmin((cv_vals - self.window_center)**2)
        window_start_conf = await extractor.extract_async(
                                # TODO/FIXME: this is the only point where we assume gromacs engine
                                outfile=os.path.join(self.workdir, "window_start.trr"),
                                traj_in=seed_trajs[-1],
                                idx=idx,
                                )
        return window_start_conf

    async def generate_trajectories(self):
        # return all trajectories by first doing the "first_trajectories", then
        # reset the propagator to n_steps_total and finally run until it is done
        if self._all_trajectories is not None:
            return self._all_trajectories
        if self._first_trajectories is None:
            await self.generate_first_trajectories()
        # first reset the number of steps!
        self._propagator.n_steps = self.n_steps_total
        # little hack to enable restart/continuation with new number of steps
        trajectories = await self._propagator.propagate(starting_configuration=None,
                                                        workdir=self.workdir,
                                                        deffnm="us",
                                                        continuation=True,
                                                        )
        print(f"Window at {self.window_center}: All trajectories done.")
        self._all_trajectories = trajectories
        return trajectories


In [ ]:
#TODO: set this to a convenient path
workdir = "."
nsteps_initial = 1000
nsteps_total = 10000

# Define windows along progress coordinate (the psi-angle of ala)
# state alpha_R: -50 degree < psi < 30 degree
# state C7_eq: 120 degree < psi < 200 degree
# Note that window_centers must be an ordered list of floats,
# starting with the window closest to the initial configuration(s)
window_centers = [30, 60, 90, 120]

# Setup our progress coordinate
# import the descriptor function
cwd = os.path.abspath(os.getcwd())
# chdir to the resources folder so we can import the function
os.chdir("../resources/")
from ala_cv_funcs import descriptor_func_psi_phi
os.chdir(cwd)  # change back to the initial directory
# and wrap it
psi_phi_wrapped = asyncmd.trajectory.PyTrajectoryFunctionWrapper(descriptor_func_psi_phi)

async def psi_wrapped(traj):
    # we only need psi and we want a 1d-array
    # also we want it in degree and not radians
    deg = 180/np.pi
    return (await psi_phi_wrapped(traj))[:, 0] / deg

# Define initial configuration (we just use a configuration in alpha_R)
# TODO: a few words on this somewhat weird function (that mirrors what we will
#       get from UmbrellaSamplingWindow.await_first_trajectories)
async def get_initial_confs():
    return [asyncmd.Trajectory(
                trajectory_files="../resources/capped_alanine_dipeptide/conf_in_alphaR.trr",
                structure_file="../resources/capped_alanine_dipeptide/conf.gro",
                )
            ]

# Engine definition
engine_kwargs_base = {gro_file="../resources/capped_alanine_dipeptide/conf.gro",
                      top_file="../resources/capped_alanine_dipeptide/topol_amber99sbildn.top",
                      ndx_file="../resources/capped_alanine_dipeptide/index.ndx",
                      output_traj_type="XTC",
                      }
mdconfig_file = "../resources/capped_alanine_dipeptide/md.mdp"
engine_cls = asyncmd.gromacs.GmxEngine
walltime_per_part = 1/(600)  # 10 s expressed in units of hours


# loop over window centers: setup pull-code in mdp,
# each window gets the first_trajectories of the previous window via seed_traj_gen
# put the initialized UmbrellaSamplingWindow objects into the windows list
windows: list[UmbrellaSamplingWindow] = []
# setup for the first window (we need a function to "generate" the seed trajectories)
seed_traj_gen = get_initial_confs
for wc in window_centers:
    # prepare the mdp (and the engine_kwargs containing it) with the correct window center
    engine_kwargs = engine_kwargs_base.copy()
    mdconfig = asyncmd.gromacs.MDP(mdconfig_file)  # load our standard mdp file
    # setup gromacs pull-code for the current window
    mdconfig["pull"] = "yes"
    mdconfig["pull-nstxout"] = 1000
    mdconfig["pull-nstfout"] = 1000
    mdconfig["pull-ngroups"] = 4  # we have 4 atoms/groups per dihedral
    mdconfig["pull-ncoords"] = 1  # we build 1 pull coord from the 4 groups
    # our 4 atoms defining the ψ dihedral
    mdconfig["pull-group1-name"] = "psi_at1"
    mdconfig["pull-group2-name"] = "psi_at2"
    mdconfig["pull-group3-name"] = "psi_at3"
    mdconfig["pull-group4-name"] = "psi_at4"
    mdconfig["pull-coord1-type"] = "umbrella"  # make it a harmonic potential
    mdconfig["pull-coord1-geometry"] = "dihedral"  # around the dihedral
    mdconfig["pull-coord1-groups"] = ["1", "2", "2", "3", "3", "4"]  # defined by our 4 atoms
    # TODO: this force-constant might be quit stiff compared to our window spacing, lets see :)
    mdconfig["pull-coord1-k"] = "110"  # force constant in kJ/ (mol * rad**2)
    mdconfig["pull-coord1-init"] = round(wc, 2) # reference angle in degree
    # IMPORTANT: We set gen-vel=yes because we use a no-modification frame extractor
    mdconfig["gen-vel"] = "yes"
    engine_kwargs["mdconfig"] = mdconfig
    # make sure the workdir for this window exists
    window_workdir = os.path.join(workdir, f"window_center_{round(wc, 2)}")
    os.makedirs(window_workdir, exist_ok=True)
    # now initialize the window object
    us_window = UmbrellaSamplingWindow(
                        window_center=wc,
                        seed_trajectories_generator=seed_traj_gen,
                        workdir=window_workdir,
                        progress_cv=psi_wrapped,
                        n_steps_initial=nsteps_initial,
                        n_steps_total=nsteps_total,
                        engine_cls=engine_cls,
                        engine_kwargs=engine_kwargs,
                        walltime_per_part=walltime_per_part,
                        )
    windows += [us_window]
    seed_traj_gen = us_window.await_first_trajectories

# now run all windows at once
all_trajectories = await asyncio.gather(*(w.generate_trajectories() for w in windows))